<a href="https://colab.research.google.com/github/vnitbrajesh-svg/Nlpprj/blob/main/IndianMarketScrapperRSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

pip install feedparser beautifulsoup4 pandas nltk

In [ ]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta
import time
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from bs4 import BeautifulSoup

# Download VADER lexicon (run once)
nltk.download('vader_lexicon', quiet=True)

def capture_multiple_rss_sentiment(days_back=15):
    # List of various Indian Market RSS Feeds
    rss_urls1 = [
        "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms", # Economic Times Markets
        "https://www.livemint.com/rss/markets",                                 # LiveMint Markets
        "https://www.business-standard.com/rss/markets-101.rss"                 # Business Standard Markets
    ]
    rss_urls = [
        "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
        "https://www.livemint.com/rss/markets",
        "https://www.business-standard.com/rss/markets-101.rss",
        "https://www.moneycontrol.com/rss/marketreports.xml",
        "https://www.thehindubusinessline.com/markets/feeder/default.rss",
        "https://www.financialexpress.com/market/feed/"
    ]



    data = []
    date_limit = datetime.now() - timedelta(days=days_back)
    sia = SentimentIntensityAnalyzer()

    for url in rss_urls:
        print(f"Fetching from: {url}")
        feed = feedparser.parse(url)

        for entry in feed.entries:
            # 1. Extract Text & Clean HTML
            title = entry.get('title', '')
            raw_summary = entry.get('summary', '')

            # Some RSS descriptions contain raw HTML (like image tags); this strips it to pure text
            clean_summary = BeautifulSoup(raw_summary, "html.parser").get_text(separator=" ", strip=True)
            full_text = f"{title}. {clean_summary}"

            # 2. Safely Parse the Publication Date
            if 'published_parsed' in entry and entry.published_parsed:
                pub_date = datetime.fromtimestamp(time.mktime(entry.published_parsed))
            else:
                pub_date = datetime.now() # Fallback if no date is provided

            # 3. Filter for the target timeframe
            if pub_date >= date_limit:
                # 4. Analyze Sentiment
                sentiment_scores = sia.polarity_scores(full_text)
                compound_score = sentiment_scores['compound']

                # Categorize
                if compound_score >= 0.05:
                    sentiment = 'Positive'
                elif compound_score <= -0.05:
                    sentiment = 'Negative'
                else:
                    sentiment = 'Neutral'

                # 5. Store the data
                data.append({
                    'Source': url.split('/')[2].replace('www.', ''), # Extract domain name cleanly
                    'Date': pub_date.strftime("%Y-%m-%d %H:%M:%S"),
                    'Text': full_text,
                    'Sentiment': sentiment,
                    'Score': compound_score
                })

    # Convert to DataFrame
    df = pd.DataFrame(data)

    # Remove any duplicate articles based on the text to keep the dataset clean
    if not df.empty:
        df = df.drop_duplicates(subset=['Text'])
        # Sort by date, newest first
        df = df.sort_values(by='Date', ascending=False)

    return df

# Run the pipeline
df_market_sentiment = capture_multiple_rss_sentiment(days_back=15)

if df_market_sentiment is not None and not df_market_sentiment.empty:
    filename = "indian_market_sentiment_multi.csv"

    # Check if the file already exists to append, or create new
    try:
        existing_df = pd.read_csv(filename)
        combined_df = pd.concat([existing_df, df_market_sentiment]).drop_duplicates(subset=['Text'])
        combined_df.to_csv(filename, index=False)
        print(f"\nAppended new data. Total records now: {len(combined_df)}")
    except FileNotFoundError:
        df_market_sentiment.to_csv(filename, index=False)
        print(f"\nCreated new file. Total records: {len(df_market_sentiment)}")

    print("\nPreview of the latest data:")
    print(df_market_sentiment[['Source', 'Sentiment', 'Text']].head())
else:
    print("No data captured.")

Fetching from: https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms
Fetching from: https://www.livemint.com/rss/markets
Fetching from: https://www.business-standard.com/rss/markets-101.rss
Fetching from: https://www.moneycontrol.com/rss/marketreports.xml
Fetching from: https://www.thehindubusinessline.com/markets/feeder/default.rss
Fetching from: https://www.financialexpress.com/market/feed/

Appended new data. Total records now: 145

Preview of the latest data:
                      Source Sentiment  \
85  thehindubusinessline.com   Neutral   
50              livemint.com   Neutral   
86  thehindubusinessline.com  Negative   
87  thehindubusinessline.com  Positive   
88  thehindubusinessline.com  Negative   

                                                 Text  
85  LPG tankers from Iran, Saudi Arabia unload car...  
50  Defying the sell-off: 5 stocks holding firm in...  
86  India buys oil from Iran for first time in 7 y...  
87  Govt says fuel supplies sufficient d